# WeatherAUS 数据集 Governance 实验

本 notebook 用于运行第二篇 governance 框架在 WeatherAUS 数据集上的实验。

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("当前项目目录：", PROJECT_ROOT)


当前项目目录： e:\yan\组\三支决策\机器学习\BT_TWD_v2


In [2]:
CONFIG_PATH = "configs/weatherAUS_bttwd.yaml"

# 是否运行消融实验：
# True：运行 no_cp 和 no_progressive 两个消融；False：只运行 full，跳过消融单元。
RUN_ABLATION = False

print("当前配置文件：", CONFIG_PATH)
print("运行消融实验：", RUN_ABLATION)


当前配置文件： configs/weatherAUS_bttwd.yaml
运行消融实验： False


## 结果摘要工具函数


In [3]:
import pandas as pd
from pathlib import Path
from IPython.display import display

output_root = Path("outputs/governance")

def show_governance_summary(mode: str) -> None:
    """Read and display the current dataset summary for one run mode."""
    config_stem = Path(CONFIG_PATH).stem
    print(f"\n===== {mode} results =====")

    dataset_summary = output_root / mode / "dataset_summary.csv"
    fold_summary = output_root / mode / config_stem / "fold_summary.csv"
    sample_records = output_root / mode / config_stem / "sample_records.csv"

    if dataset_summary.exists():
        print("Reading:", dataset_summary)
        summary_df = pd.read_csv(dataset_summary)
        if "dataset_name" in summary_df.columns:
            dataset_row = summary_df[summary_df["dataset_name"].astype(str) == config_stem]
            if dataset_row.empty:
                available = ", ".join(summary_df["dataset_name"].astype(str).tolist())
                print(f"No row for {config_stem}; available datasets: {available}")
            else:
                display(dataset_row.reset_index(drop=True))
        else:
            print("dataset_summary.csv has no dataset_name column; cannot filter by dataset.")
    else:
        print("Missing dataset_summary.csv:", dataset_summary)

    if fold_summary.exists():
        print("Reading:", fold_summary)
        display(pd.read_csv(fold_summary))
    else:
        print("Missing fold_summary.csv:", fold_summary)

    if sample_records.exists():
        print("Sample records:", sample_records)
    else:
        print("Missing sample_records.csv:", sample_records)


## 运行完整 governance 实验


In [4]:
import subprocess

cmd = [
    sys.executable,
    "scripts/run_governance_experiments.py",
    "--config",
    CONFIG_PATH,
]

print("运行命令：", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)

print("标准输出：")
print(result.stdout)

print("错误输出：")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"实验运行失败，返回码：{result.returncode}")

show_governance_summary("full")


运行命令： d:\Anaconda3\python.exe scripts/run_governance_experiments.py --config configs/weatherAUS_bttwd.yaml
标准输出：
【INFO】【2026-05-18 19:50:52】【数据加载】文本表格 E:\yan\组\三支决策\机器学习\BT_TWD_v2\data\weather\weatherAUS.csv 已读取，样本数=145460，列数=23
【INFO】【2026-05-18 19:50:52】【数据加载】3267 条标签无法映射，占比=2.25%，正负类已指定且未开启 dropna_target，已自动删除这些样本
【INFO】【2026-05-18 19:50:52】【数据加载】标签列 RainTomorrow 已处理完成：dropna_target=False, 丢弃样本=3267, 最终样本数=142193, 正类比例=22.42%
【INFO】【2026-05-18 19:50:53】【数据集信息】名称=weatherAUS，样本数=142193，目标列=RainTomorrow，正类比例=22.42%
【INFO】【2026-05-18 19:50:53】【预处理】连续特征=16个，类别特征=5个
【INFO】【2026-05-18 19:50:54】【预处理】编码后维度=110
【INFO】【2026-05-18 19:50:54】【governance】weatherAUS_bttwd 第 1/5 折
【INFO】【2026-05-18 19:50:57】【桶树】列 Humidity3pm 出现未知取值，1975 条记录记为 unknown
【INFO】【2026-05-18 19:50:57】【桶树】已为样本生成桶ID，共 478 个组合
【INFO】【2026-05-18 19:50:57】【桶树】列 Humidity3pm 出现未知取值，929 条记录记为 unknown
【INFO】【2026-05-18 19:50:57】【桶树】已为样本生成桶ID，共 418 个组合
【INFO】【2026-05-18 19:51:50】【governance】weak bucket resolver：strong=141，weak=560
【

,dataset_name,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,closure_rate,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,weatherAUS_bttwd,0.284972,0.799527,0.661199,0.828297,0.282384,0.771594,0.656756,0.854332,1.0,...,0.991968,0.014124,0.985876,0.008032,0.988391,0.412935,0.575456,0.977844,349.0,6.0


Reading: outputs\governance\full\weatherAUS_bttwd\fold_summary.csv


,dataset_name,fold_id,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,weatherAUS_bttwd,1,0.284715,0.799607,0.661843,0.829073,0.283220,0.770065,0.653821,0.852808,...,0.981481,0.029851,0.970149,0.018519,0.975207,0.446281,0.528926,0.951631,65.0,3.0
1,weatherAUS_bttwd,2,0.286438,0.798503,0.659687,0.827420,0.284556,0.768437,0.651741,0.852315,...,0.983051,0.015152,0.984848,0.016949,0.984000,0.472000,0.512000,0.967899,65.0,3.0
2,weatherAUS_bttwd,3,0.289567,0.796183,0.656895,0.826330,0.284398,0.771830,0.656428,0.853687,...,1.000000,0.017241,0.982759,0.000000,0.989362,0.382979,0.606383,0.982759,57.0,0.0
3,weatherAUS_bttwd,4,0.282123,0.801814,0.662944,0.828082,0.281349,0.772879,0.658186,0.854490,...,1.000000,0.010638,0.989362,0.000000,0.993289,0.369128,0.624161,0.989362,93.0,0.0
4,weatherAUS_bttwd,5,0.282017,0.801527,0.664625,0.830579,0.278395,0.774759,0.663604,0.858359,...,1.000000,0.000000,1.000000,0.000000,1.000000,0.394737,0.605263,1.000000,69.0,0.0


Sample records: outputs\governance\full\weatherAUS_bttwd\sample_records.csv


## 运行无 CP 消融


In [5]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-cp-validation",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无 CP 消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_cp")
else:
    print("RUN_ABLATION=False，跳过无 CP 消融。")


RUN_ABLATION=False，跳过无 CP 消融。


## 运行无渐进更新消融


In [6]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-progressive-update",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无渐进更新消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_progressive")
else:
    print("RUN_ABLATION=False，跳过无渐进更新消融。")


RUN_ABLATION=False，跳过无渐进更新消融。
